In [ ]:
# Run once: install pyyaml if not already installed
!pip install pyyaml

In [3]:
#Finding where am I
import os
os.environ['USE_LOCALSTACK'] = '1'
os.environ['LOCALSTACK_URL'] = 'http://localhost:4566'
# os.chdir('/config/workspace/python/pythonawsproject')
# print(os.getcwd())

# Now import (after chdir so Python can find the network package)
from network.state import load_state
print(load_state())

{}


In [4]:
#What's the plan yo!
import yaml

with open('config/network.yaml', 'r') as f:
    config = yaml.safe_load(f) #//putting all the values from load state to config

print(config['vpc'])
print(config['security_groups'])

{'name': 'dev-vpc', 'cidr': '10.0.0.0/16', 'region': 'ap-south-1'}
[{'name': 'dev-web-sg', 'description': 'Allow SSH, HTTP, HTTPS inbound', 'ingress': [{'protocol': 'tcp', 'port': 22, 'cidr': '0.0.0.0/0', 'description': 'SSH access'}, {'protocol': 'tcp', 'port': 80, 'cidr': '0.0.0.0/0', 'description': 'HTTP'}, {'protocol': 'tcp', 'port': 443, 'cidr': '0.0.0.0/0', 'description': 'HTTPs'}], 'egress': [{'protocol': 'tcp', 'port': 80, 'cidr': '0.0.0.0/0', 'description': 'HTTP'}]}]


In [5]:
##Now it can work
from network.vpc import create_vpc

vpc_id = create_vpc(config)
print(f"Result: {vpc_id}")

Created VPC: vpc-a3ef777a
VPC is now avilable
Tagged VPC with: ['Name', 'ManagedBy', 'Layer', 'Environment']
State saved
Result: vpc-a3ef777a


In [6]:
from network.subnets import create_subnet
##We are utting full config python file here for execution
result = create_subnet(config)
print(result)

[{'name': 'public-subnet-az1', 'cidr': '10.0.1.0/24', 'az': 'ap-south-1a'}, {'name': 'public-subnet-az2', 'cidr': '10.0.2.0/24', 'az': 'ap-south-1b'}, {'name': 'private-subnet-az1', 'cidr': '10.0.3.0/24', 'az': 'ap-south-1a'}, {'name': 'private-subnet-az2', 'cidr': '10.0.4.0/24', 'az': 'ap-south-1b'}]
{'Subnet': {'AvailabilityZoneId': 'aps1-az1', 'MapCustomerOwnedIpOnLaunch': False, 'OwnerId': '000000000000', 'AssignIpv6AddressOnCreation': False, 'Ipv6CidrBlockAssociationSet': [], 'Tags': [], 'SubnetArn': 'arn:aws:ec2:ap-south-1:000000000000:subnet/subnet-53840dce', 'EnableDns64': False, 'SubnetId': 'subnet-53840dce', 'State': 'available', 'VpcId': 'vpc-a3ef777a', 'CidrBlock': '10.0.1.0/24', 'AvailableIpAddressCount': 251, 'AvailabilityZone': 'ap-south-1a', 'DefaultForAz': False, 'MapPublicIpOnLaunch': False}, 'ResponseMetadata': {'RequestId': '346f5495-2de3-4820-bfeb-5ad6d670ce3e', 'HTTPStatusCode': 200, 'HTTPHeaders': {'content-length': '862', 'content-type': 'application/xml', 'date

In [7]:
from network.internet_gateway  import creating_igw
from network.state import load_state, save_state
from network.aws_client import get_client

igw_id = creating_igw(config)
print(f"Result: {igw_id}")
# ec2 = get_client('ec2')
# response = ec2.describe_internet_gateways(
#     Filters=[{'Name': 'attachment.vpc-id', 'Values': [vpc_id]}]
# )

# print(vpc_id)

# response = ec2.describe_internet_gateways()
# print(response['InternetGateways'])

# igw_id = response['InternetGateways'][0]['InternetGatewayId']
# print(f"Found exsisting IGW: {igw_id}")

# ec2.attach_internet_gateway(InternetGatewayId=igw_id, VpcId=vpc_id)
# print(f"Attached {igw_id} to {vpc_id}")

# ##Saving and loading of state file
# state = load_state()
# state['igw_id'] = igw_id
# save_state(state)
# print("Saved State")

# igw_id = creating_igw(config)
# print(f"Result: {igw_id}")

Created IGW: igw-db5fac71
Attachment IGW to VPC: vpc-a3ef777a
Tagged  IGW: dev-igw
Saved IGW to State
Result: igw-db5fac71


In [8]:
from network.route_tables import create_route_tables

result = create_route_tables(config)

Created public route table: rtb-3bed5d1f
Added route: '0.0.0.0/0' rtb-3bed5d1f
Public route table already exists: rtb-3bed5d1f
Created private route table: rtb-d356522a
Saved private route table to state
  Associated private-subnet-az1 → private route table
  Associated private-subnet-az2 → private route table
Route tables done.


In [9]:
from network.nat_gateway import create_nat_gateway

result = create_nat_gateway(config)

Allocated Elastic IP: eipalloc-3e379cd84bb262e40
Created NAT Gateway: nat-ac2c06f6d33ee1538
Waiting for NAT Gateway to become available (1-2 min)...
NAT Gateway is now available
Saved NAT Gateway to state
Added route: 0.0.0.0/0 → nat-ac2c06f6d33ee1538 in private route table
NAT Gateway done.


In [10]:
from network.state import load_state, save_state

state = load_state()
del state['nat_id']
del state['nat_eip_alloc_id']

In [11]:
import os
os.chdir('/workspaces/floci-boto-3-python')
print(os.listdir('network/'))

['ec2_instance.py', 'key_pair.py', 'state.py', 'main.py', '__pycache__', 'nat_gateway.py', 'security_groups.py', 'internet_gateway.py', 'subnets.py', '__init__.py', 'route_tables.py', 'aws_client.py', 'planner.py', 'vpc.py']


In [12]:
from network.security_groups import create_security_groups
create_security_groups(config)

Created Security Group: sg-9a86baca8c66d60a2 (dev-web-sg)
  Added 3 ingress rules
  Revoked default egress (allow-all) rule
  Tagged + saved: sg_dev-web-sg = sg-9a86baca8c66d60a2
Security Groups done.


{'vpc_id': 'vpc-a3ef777a',
 'vpc_name': 'dev-vpc',
 'public-subnet-az1': 'subnet-53840dce',
 'public-subnet-az2': 'subnet-5ea54c75',
 'private-subnet-az1': 'subnet-325f9ea9',
 'private-subnet-az2': 'subnet-e299983b',
 'igw_id': 'igw-db5fac71',
 'public_rt_id': 'rtb-3bed5d1f',
 'public-subnet-az1_rt_assoc': 'rtbassoc-e7c81013',
 'public-subnet-az2_rt_assoc': 'rtbassoc-2772f8a2',
 'private_rt_id': 'rtb-d356522a',
 'private-subnet-az1_rt_assoc': 'rtbassoc-d4118d05',
 'private-subnet-az2_rt_assoc': 'rtbassoc-da779353',
 'nat_eip_alloc_id': 'eipalloc-3e379cd84bb262e40',
 'nat_id': 'nat-ac2c06f6d33ee1538',
 'private_nat_route': True,
 'sg_dev-web-sg': 'sg-9a86baca8c66d60a2'}

In [17]:
# from network.key_pair import create_key_pair
# create_key_pair(config)

import importlib
import network.key_pair

importlib.reload(network.key_pair)
key_name = network.key_pair.create_key_pair(config)
print("Returned key:", key_name)

Key pair 'dev-key' found in AWS/Floci without local .pem. Recreating...
Created key pair 'dev-key' and saved to keys/dev-key.pem
Returned key: dev-key


In [18]:

from network.ec2_instance import create_ec2_instance
instance_id = create_ec2_instance(config)
print(f"Result: {instance_id}")

Launched EC2 instance: i-153aef6d5870bcf5f
Waiting for instance to reach 'running' state...
Instance is now running.
Save EC@ instance to state: i-153aef6d5870bcf5f
Result: i-153aef6d5870bcf5f


In [19]:
from network.planner import run_planner
plan = run_planner()

🔍 Verifying resources against AWS...

  ✅ No drift — state matches AWS.

  EXECUTION PLAN
  ✅ vpc                  "dev-vpc" → EXISTS (vpc-a3ef777a...)
  ✅ subnet               "public-subnet-az1" → EXISTS (subnet-53840dce...)
  ✅ subnet               "public-subnet-az2" → EXISTS (subnet-5ea54c75...)
  ✅ subnet               "private-subnet-az1" → EXISTS (subnet-325f9ea9...)
  ✅ subnet               "private-subnet-az2" → EXISTS (subnet-e299983b...)
  ✅ internet_gateway     "dev-igw" → EXISTS (igw-db5fac71...)
  ✅ nat_gateway          "dev-nat" → EXISTS (nat-ac2c06f6d33ee153...)
  ✅ route_table          "public-route-table" → EXISTS (rtb-3bed5d1f...)
  ✅ route_table          "private-route-table" → EXISTS (rtb-d356522a...)
  ✅ security_group       "dev-web-sg" → EXISTS (sg-9a86baca8c66d60a2...)
  0 to create, 0 to delete, 10 unchanged

  ✅ Infrastructure matches desired state. Nothing to do.


In [20]:
from network.main import command_plan
command_plan(config)


 Scanning the  battlefiend that is config file
🔍 Verifying resources against AWS...

  ✅ No drift — state matches AWS.

  EXECUTION PLAN
  ✅ vpc                  "dev-vpc" → EXISTS (vpc-a3ef777a...)
  ✅ subnet               "public-subnet-az1" → EXISTS (subnet-53840dce...)
  ✅ subnet               "public-subnet-az2" → EXISTS (subnet-5ea54c75...)
  ✅ subnet               "private-subnet-az1" → EXISTS (subnet-325f9ea9...)
  ✅ subnet               "private-subnet-az2" → EXISTS (subnet-e299983b...)
  ✅ internet_gateway     "dev-igw" → EXISTS (igw-db5fac71...)
  ✅ nat_gateway          "dev-nat" → EXISTS (nat-ac2c06f6d33ee153...)
  ✅ route_table          "public-route-table" → EXISTS (rtb-3bed5d1f...)
  ✅ route_table          "private-route-table" → EXISTS (rtb-d356522a...)
  ✅ security_group       "dev-web-sg" → EXISTS (sg-9a86baca8c66d60a2...)
  0 to create, 0 to delete, 10 unchanged

  ✅ Infrastructure matches desired state. Nothing to do.


[{'type': 'vpc', 'name': 'dev-vpc', 'action': 'NO-OP', 'id': 'vpc-a3ef777a'},
 {'type': 'subnet',
  'name': 'public-subnet-az1',
  'action': 'NO-OP',
  'id': 'subnet-53840dce'},
 {'type': 'subnet',
  'name': 'public-subnet-az2',
  'action': 'NO-OP',
  'id': 'subnet-5ea54c75'},
 {'type': 'subnet',
  'name': 'private-subnet-az1',
  'action': 'NO-OP',
  'id': 'subnet-325f9ea9'},
 {'type': 'subnet',
  'name': 'private-subnet-az2',
  'action': 'NO-OP',
  'id': 'subnet-e299983b'},
 {'type': 'internet_gateway',
  'name': 'dev-igw',
  'action': 'NO-OP',
  'id': 'igw-db5fac71'},
 {'type': 'nat_gateway',
  'name': 'dev-nat',
  'action': 'NO-OP',
  'id': 'nat-ac2c06f6d33ee1538'},
 {'type': 'route_table',
  'name': 'public-route-table',
  'action': 'NO-OP',
  'id': 'rtb-3bed5d1f'},
 {'type': 'route_table',
  'name': 'private-route-table',
  'action': 'NO-OP',
  'id': 'rtb-d356522a'},
 {'type': 'security_group',
  'name': 'dev-web-sg',
  'action': 'NO-OP',
  'id': 'sg-9a86baca8c66d60a2'}]

In [ ]:
from network.main import command_destroy
command_destroy(config)


 RETREAT ORDER - Demolishing infrastructure ...

====== Terminate EC2 Instance ======
. Waiting for i-153aef6d5870bcf5f to terminate...
====== Delete Key Pair ======
===== Destroying NAT Gateway =====
  Waiting for NAT nat-ac2c06f6d33ee1538 to die...


In [ ]:
command_plan(config)